In [1]:
from absl import logging

import tensorflow as tf

import tensorflow_hub as hub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
model = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return model(input)

2025-10-11 11:06:06.049429: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-11 11:06:06.049643: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 11:06:06.079373: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 11:06:07.170256: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different compu

module https://tfhub.dev/google/universal-sentence-encoder/4 loaded


In [2]:
books = pd.read_csv("data/processed/books.csv")
genres = pd.read_csv("data/processed/genres.csv")
reviews = pd.read_csv("data/processed/reviews.csv")
ian_reviews = pd.read_csv("data/processed/ian_reviews.csv")

In [3]:
# limit to books that have a link
books = books[books["link"].notna()]
books.loc[:,"title"] = books["title"].fillna("")
books.loc[:,"author"] = books["author"].fillna("")
books.loc[:,"description"] = books["description"].fillna("")
reviews.loc[:,"review_text"] = reviews["review_text"].fillna("")

In [4]:
def embed_book_features(book):
  book_features = book["title"] + " " + book["author"] + " " + book["description"]
  book_embeddings = embed([book_features])[0]
  review_embeddings = embed_book_reviews(book["link"])
  genre_embeddings = embed_book_genres(book["link"])
  return tf.concat([book_embeddings, review_embeddings, genre_embeddings], axis=0)

# For a book, embed the text from its reviews and pool them
def embed_book_reviews(url):
  book_reviews = reviews[reviews["book_url"] == url]
  # combine review_author, review_publisher, and review_text by row
  review_features = book_reviews["review_author"] + " " + book_reviews["review_publisher"] + " " + book_reviews["review_text"]
  review_features = (
      book_reviews["review_author"].astype(str).fillna("") + " " +
      book_reviews["review_publisher"].astype(str).fillna("") + " " +
      book_reviews["review_text"].astype(str).fillna("")
  )
  review_embeddings = embed(review_features.to_list())
  return tf.reduce_mean(review_embeddings, axis=0)

def embed_book_genres(url):
  book_genres = genres[genres["link"] == url]
  genre_embeddings = embed(book_genres["genre"].tolist())
  return tf.reduce_mean(genre_embeddings, axis=0)

In [5]:
# prompt: find cosine similarity between the a book and the rest of the books

from sklearn.metrics.pairwise import cosine_similarity

def find_similar_books(book_index, books_df, book_embeddings_tensor):
  """Finds the cosine similarity between a given book and the rest of the books.

  Args:
    book_index: The index of the book to compare against.
    books_df: The pandas DataFrame containing book information.
    book_embeddings_tensor: The TensorFlow tensor containing book embeddings.

  Returns:
    A pandas DataFrame with 'title', 'author', and 'similarity' for all other books,
    sorted by similarity in descending order.
  """
  book_embedding = book_embeddings_tensor[book_index].numpy().reshape(1, -1)
  other_book_embeddings = np.delete(book_embeddings_tensor.numpy(), book_index, axis=0)

  similarities = cosine_similarity(book_embedding, other_book_embeddings)[0]

  # Create a list of books excluding the target book
  other_books = books_df.drop(books_df.index[book_index]).reset_index(drop=True)

  results_df = pd.DataFrame({
      'title': other_books['title'],
      'author': other_books['author'],
      'similarity': similarities
  })

  return results_df.sort_values(by='similarity', ascending=False)

In [6]:
# embed the fatures for each book and stack on top of each other
book_embeddings = tf.stack([embed_book_features(book) for _, book in books.iterrows()])

2025-10-11 11:06:13.833293: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [7]:
# save to drive as a numpy array
np.save("data/processed/book_embeddings.npy", book_embeddings.numpy())

In [8]:
book_embeddings = np.load("data/processed/book_embeddings.npy")
# convert to tensor
book_embeddings = tf.convert_to_tensor(book_embeddings)

In [9]:
# Example usage: Find books similar to the first book (index 0)
similar_books_df = find_similar_books(0, books, book_embeddings)
print(similar_books_df.head())

                                                   title  \
2715                The Undercurrents: A Story of Berlin   
14359       These Truths: A History of the United States   
6871                                     Distant Fathers   
14834  Know Thyself: Western Identity from Classical ...   
6332   Accidental Gods: On Men Unwittingly Turned Divine   

                                author  similarity  
2715                       Kirsty Bell    0.629759  
14359                      Jill Lepore    0.600522  
6871   Marina Jarre, tr. Ann Goldstein    0.596933  
14834                Ingrid Rossellini    0.596927  
6332                  Anna Della Subin    0.595585  


In [10]:
# prompt: find average embeddings given an array of book urls

def get_embeddings(book_urls, books_df, book_embeddings_tensor):
  """Calulates embeddings for a list of book URLs.

  Args:
    book_urls: A list of book URLs.
    books_df: The pandas DataFrame containing book information (including the 'link' column).
    book_embeddings_tensor: The TensorFlow tensor containing book embeddings, aligned with books_df.

  Returns:
    A TensorFlow tensor representing the embeddings of the specified books,
    or None if none of the URLs are found.
  """
  valid_indices = []
  for url in book_urls:
    indices = books_df[books_df['link'] == url].index.tolist()
    valid_indices.extend(indices)

  if not valid_indices:
    print("None of the provided URLs were found in the books data.")
    return None

  # Get the embeddings for the books at the valid indices
  selected_embeddings = tf.gather(book_embeddings_tensor, valid_indices)

  return selected_embeddings

# Example usage:
# Assuming you have a list of book URLs
# sample_urls = ["url1", "url2", "url3"]
# average_embed = get_average_embedding_for_urls(sample_urls, books, book_embeddings)
# if average_embed is not None:
#   print("Average embedding shape:", average_embed.shape)


In [ ]:
# prompt: use cosine similarity to find closest books to user embeddings

def find_closest_books_to_user(book_urls, ratings, books_df, book_embeddings_tensor, top_n=10):
  """Finds the books with the highest cosine similarity to a user embedding.

  Args:
    user_embedding: The TensorFlow tensor representing the user's embedding.
    books_df: The pandas DataFrame containing book information.
    book_embeddings_tensor: The TensorFlow tensor containing book embeddings, aligned with books_df.
    top_n: The number of closest books to return.

  Returns:
    A pandas DataFrame with 'title', 'author', and 'similarity' for the top_n
    closest books, sorted by similarity in descending order.
  """

  valid_indices = []
  for url in book_urls:
    indices = books_df[books_df['link'] == url].index.tolist()
    valid_indices.extend(indices)

  books_df = books_df.loc[~books_df['link'].isin(book_urls)].reset_index(drop=True)

  if not valid_indices:
    print("None of the provided URLs were found in the books data.")
    return None

  # Get the embeddings for the books at the valid indices
  selected_embeddings = tf.gather(book_embeddings_tensor, valid_indices)

  # Create weighted average embeddings with ratings
  weights = tf.convert_to_tensor(ratings, dtype=tf.float32)
  weights = weights / tf.reduce_sum(weights)
  weights = tf.expand_dims(weights, axis=1)

  weighted_embeddings = tf.multiply(selected_embeddings, weights)

  user_embedding = tf.reduce_sum(weighted_embeddings, axis=0)

  unselected_embeddings = tf.gather(book_embeddings_tensor, [i for i in range(len(book_embeddings_tensor)) if i not in valid_indices])

  user_embedding_np = user_embedding.numpy().reshape(1, -1)
  book_embeddings_np = unselected_embeddings.numpy()

  similarities = cosine_similarity(user_embedding_np, book_embeddings_np)[0]

  results_df = pd.DataFrame({
      'url': books_df['link'],
      'title': books_df['title'],
      'author': books_df['author'],
      'similarity': similarities,
      'description': books_df['description']
  })

  return results_df.sort_values(by='similarity', ascending=False).head(top_n)

# Example usage:
# Assuming you have a user embedding calculated from their liked books
# user_liked_book_urls = ["url_of_book_user_liked_1", "url_of_book_user_liked_2"]
# user_embed = get_user_embeddings(user_liked_book_urls, books, book_embeddings)
#
# if user_embed is not None:
#   closest_books_df = find_closest_books_to_user(user_embed, books, book_embeddings, top_n=5)
#   print("\nClosest books to user:")
#   print(closest_books_df)

In [24]:
urls = ian_reviews['book_url'].tolist()
ratings = ian_reviews['numeric_rating'].values
# urls = books.loc[books['link'].isin(urls) & (books['is_fiction'] == 0), 'link'].tolist()
ians_recs = find_closest_books_to_user(urls, ian_reviews['numeric_rating'], books, book_embeddings, top_n=100)
ians_recs.to_csv('data/ians_recs.csv')

In [21]:
user_embeddings = get_embeddings(urls, books, book_embeddings)

In [18]:
# prompt: perform k means clustering on the user embeddings and print out the book titles in each cluster

from sklearn.cluster import KMeans

# Convert user_embeddings to a NumPy array for KMeans
user_embeddings_np = user_embeddings.numpy()

# Perform KMeans clustering
n_clusters = 5  # You can adjust the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
clusters = kmeans.fit_predict(user_embeddings_np)

# Add cluster assignments to the original dataframe (if needed) or use indices
# Since user_embeddings are derived from a subset of books, we need to map
# the cluster indices back to the original book titles.
# First, get the indices of the books that generated the user embeddings
valid_indices = []
urls = ian_reviews.loc[ian_reviews['numeric_rating'] >= 3,"book_url"].tolist()
for url in urls:
  indices = books[books['link'] == url].index.tolist()
  valid_indices.extend(indices)

valid_indices = books.index.tolist()

# Create a list of book titles corresponding to the user embeddings
user_book_titles = books.loc[valid_indices, 'title'].tolist()

# Print books in each cluster
for i in range(n_clusters):
  print(f"\nCluster {i+1}:")
  cluster_indices = [j for j, cluster_id in enumerate(clusters) if cluster_id == i]
  cluster_book_titles = [user_book_titles[j] for j in cluster_indices]
  for title in cluster_book_titles[0:10]:
    print(f"- {title}")



Cluster 1:
- Second Life: Having a Child in the Digital Age
- Capitalism and Its Critics: A History: From the Industrial Revolution to AI
- Hope I Get Old Before I Die: Why Rock Stars Never Retire
- Matriarch: A Memoir
- Atavists: Stories
- Medicine River: A Story of Survival and the Legacy of Indian Boarding Schools
- Heaven and Hell
- The Imagined Life
- What's Left: Three Paths Through the Planetary Crisis

Cluster 2:
- The Unworthy
- Gandolfini: Jim, Tony, and the Life of a Legend
- Dianaworld: An Obsession
- The Third Reich of Dreams: The Nightmares of a Nation
- All the Other Mothers Hate Me
- The Persians
- We Pretty Pieces of Flesh
- No More Tears: The Dark Secrets of Johnson & Johnson
- A Training School for Elephants
- The Acid Queen: The Psychedelic Life and Counterculture Rebellion of Rosemary Woodruff Leary

Cluster 3:
- Foreign Fruit: A Personal History of the Orange
- Girl on Girl: How Pop Culture Turned a Generation of Women Against Themselves
- Things in Nature Merely

In [ ]:
urls = [
    "https://bookmarks.reviews/reviews/i-am-malala-the-girl-who-stood-up-for-education-and-was-shot-by-the-taliban/",
    "https://bookmarks.reviews/reviews/my-brilliant-friend/",
    "https://bookmarks.reviews/reviews/educated/",
    "https://bookmarks.reviews/reviews/marilla-of-green-gables/",
    "https://bookmarks.reviews/reviews/the-honey-bus-a-memoir-of-loss-courage-and-a-girl-saved-by-bees-original/",
    "https://bookmarks.reviews/reviews/klara-and-the-sun/",
    "https://bookmarks.reviews/reviews/the-story-of-a-new-name/",
    "https://bookmarks.reviews/reviews/those-who-leave-and-those-who-stay/",
    "https://bookmarks.reviews/reviews/story-of-the-lost-child/"
]
ratings = np.array([4, 3, 3, 4, 3, 3, 4, 3, 3])
find_closest_books_to_user(urls, ratings, books, book_embeddings, top_n=100)

,title,author,similarity
12480,The Lowland,Jhumpa Lahiri,0.803435
9190,A Mercy,Toni Morrison,0.801753
9633,The Last Romantics,Tara Conklin,0.800839
11260,The Maid's Version,Daniel Woodrell,0.794930
6935,Inseparable: A Never-Before-Published Novel,"Simone De Beauvoir, Tr. Sandra Smith",0.791118
...,...,...,...
10551,Small Days and Nights,Tishani Doshi,0.757964
8111,Three Summers,"Margarita Liberaki, Trans. by Karen Van Dyck",0.757958
7831,What Comes After,Joanne Tompkins,0.757596
8668,Orange World and Other Stories,Karen Russell,0.757557


In [ ]:
urls = books.loc[books['link'].isin(urls) & (books['is_fiction'] == 0), 'link'].tolist()
find_closest_books_to_user(urls, books, book_embeddings, top_n=100)